### Importing Essential Modules

In [ ]:
import torch
import time
import librosa
import os
import random

from torch import nn
from torchvision.transforms import Compose, ToTensor
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.io import wavfile
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report


import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

### Loading Dataset

In [ ]:
ALL_CLASSES = list("abcdefghijklmnopqrstuvwxyz0123456789")  # 36 classes
label_encoder = LabelEncoder()
label_encoder.fit(ALL_CLASSES)
CLASS_NAMES = ALL_CLASSES


class AudioAugment:
    """Apply random augmentations to mel spectorgrams during training"""
    def __call__(self, mel):
        
        """Time Masking: Randomly zero out time strips"""
        if random.random() > 0.5:
            timeMask = random.randint(1, max(1, mel.shape[1] // 8))
            timeStart = random.randint(0, mel.shape[1] - timeMask)
            mel[:, timeStart:timeStart + timeMask] = mel.min()

        """Frequency Masking: Rnadomly zero out frequency bands"""
        if random.random() > 0.5:
            frequencyMask = random.randint(1, max(1, mel.shape[0] // 8))
            frequencyStart = random.randint(0, mel.shape[0] - frequencyMask)
            mel[frequencyStart:frequencyStart + frequencyMask, :] = mel.min()

        """Random Noise"""
        if random.random() > 0.5:
            noise = np.random.normal(0, 0.01, mel.shape).astype(np.float32)
            mel = mel + noise

        return mel


"""This transform the audio signal into mel-spectrogram, which will be the input for CoATnet model to processed"""
class ToMelSpectrogram:
    """"Augmentation is only APPLIED during training only to prevent overfitting"""
    def __init__(self, augment=False, targetFrames=99, sampleRates=44100):
        self.augment = augment 
        self.targetFrames = targetFrames
        self.sampleRates = sampleRates
        
    def __call__(self, samples):
        """Random pitch shift before spectrogram (training only)"""
        if self.augment and random.random() > 0.5:
            n_steps = random.uniform(-2, 2)
            samples = librosa.effects.pitch_shift(samples, sr=self.sampleRates, n_steps=n_steps)

        """Random time stretch (training only)"""
        if self.augment and random.random() > 0.5:
            rate = random.uniform(0.9, 1.1)
            samples = librosa.effects.time_stretch(samples, rate=rate)
            # Trim or pad back to expected length
            originalLen = int(self.sampleRates * 1.0)
            if len(samples) > originalLen:
                samples = samples[:originalLen]
            else:
                samples = np.pad(samples, (0, originalLen - len(samples)), mode='constant')
        
        mel = librosa.feature.melspectrogram(y=samples, sr=self.sampleRates, n_mels=64, n_fft=1024, hop_length=225)
        """Convert power spectrogram to dB scale — better for model training"""
        melDb = librosa.power_to_db(mel, ref=np.max)

        """Fix Melotrogram Length"""
        currentFrames = melDb.shape[1]
        if currentFrames < self.targetFrames:
            padWidth = self.targetFrames - currentFrames
            melDb = np.pad(melDb, ((0, 0), (0, padWidth)), mode='constant', constant_values=melDb.min())
        elif currentFrames > self.targetFrames:
            melDb = melDb[:, :self.targetFrames]
            
        if self.augment:
            melDb = AudioAugment()(melDb)
        return melDb


"""Defining a dataset class to store audio data to train the model"""
class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, dataDir, transform=None):
        """Setting the directory that contains audio data to be trained on"""
        self.dataDir = dataDir
        self.transform = transform
        self.samples = []
        for dirpath, _, filenames in os.walk(self.dataDir):
            for filename in sorted(filenames):
                if filename.endswith(".wav"):
                    filepath = os.path.join(dirpath, filename)
                    label = os.path.basename(dirpath)
                    self.samples.append((filepath, label))
        print(f"Dataset loaded: {len(self.samples)} samples across "
              f"{len(set(l for _, l in self.samples))} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filepath, label = self.samples[idx]

        waveform, _ = librosa.load(filepath, sr = None, duration = 1.0, mono = True)
        if self.transform:
            waveform = self.transform(waveform)

        encodedLabel = label_encoder.transform([label])[0]
        label  = torch.tensor(encodedLabel, dtype=torch.long)
        return waveform, label

    @property
    def targets(self):
        """Returns list of all labels — required for stratified train/val split"""
        return [label for _, label in self.samples]



"""Getting the largest number of frames and average sample rates out of all the audio in the dataset"""

LargestMelframesForCombinedDataset = 0
AllMelFrames = []
CombinedAudioSampleRates = []
for dirpath, _, filenames in os.walk("CombinedDataset"):
    for filename in sorted(filenames):
        if filename.endswith(".wav"):
            filepath = os.path.join(dirpath, filename)
            # print(f"Reading WAV File at path {filepath}")
            signals, sampleRate = librosa.load(filepath, mono=True)
            mel = librosa.feature.melspectrogram(y=signals, sr=sampleRate, n_fft=1024, hop_length=225)
            melFrames = mel.shape[1]
            AllMelFrames.append(melFrames)
            # print(f"Total Audio frames: {len(signals)}")
            # print(f"Sample Rate: {sampleRate} Hz")
            # print(f"Total Meltrogram time frames: {melFrames}")
            LargestMelframesForCombinedDataset = max(LargestMelframesForCombinedDataset, melFrames)
            CombinedAudioSampleRates.append(sampleRate)
    
print(f"Final largest Meltrogram nframes in Combined Dataset: {LargestMelframesForCombinedDataset}")
MeanCombinedAudioSampleRate = sum(CombinedAudioSampleRates) // len(CombinedAudioSampleRates)
print(f"Mean sample rate across all audios in Combined Audio Dataset: {MeanCombinedAudioSampleRate}")

In [ ]:
"""We will use the transformation to convert the audio into Mel spectrogram"""
trainingTransform = Compose([ToMelSpectrogram(augment=True, targetFrames=LargestMelframesForCombinedDataset, sampleRates=MeanCombinedAudioSampleRate), ToTensor()])
validationTransform = Compose([ToMelSpectrogram(augment=False, targetFrames=LargestMelframesForCombinedDataset, sampleRates=MeanCombinedAudioSampleRate), ToTensor()])


AudioTrainDataSet = AudioDataset("CombinedDataset", transform=trainingTransform) 
AudioValDataSet = AudioDataset("CombinedDataset", transform=validationTransform)



# Check total count
print(f"Total Combined samples : {len(AudioTrainDataSet)}")
# Check (path, label) pairs
for path, label in AudioTrainDataSet.samples:
    print(f"  {label}  ←  {path}")
# Check direct indexing works correctly
for i in range(len(AudioTrainDataSet) - 1):
    waveform, label = AudioTrainDataSet[i]
    print(f"  dataset[{i}] → label='{label}', shape={waveform.shape}")

### Data Loader

In [ ]:
BATCHSIZE = 16

"""Splitting Dataset 80% train and 20% validation"""
trainIdx, valIdx = train_test_split(range(len(AudioTrainDataSet)), test_size=0.2, stratify=AudioTrainDataSet.targets, random_state=42)
AudioTrainSet = Subset(AudioTrainDataSet, trainIdx)
AudioValSet = Subset(AudioValDataSet, valIdx)


"""Loading Batch of samples for training and validation"""
AudioTrainLoader = DataLoader(dataset=AudioTrainSet, batch_size=BATCHSIZE, shuffle=True)
AudioValLoader = DataLoader(dataset=AudioValSet, batch_size=BATCHSIZE, shuffle=False)

### Building The CoATnet model

In [ ]:
class CoAtNet(nn.Module):
    """A hybrid ML architecture combined Convolution layers and Transformers"""
    
    def __init__(self, num_classes=36, dropout=0.3):
        """We only train to recognize keystroke a-z0-9, adding regulatization to reduce overfitting"""
        super(CoAtNet, self).__init__()

        # Convolutional part
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1), # Takes 1 input channel (grayscale mel spectrogram) and output 32 features map, padding to keep spatial dimension the same 
            nn.BatchNorm2d(32), # Normalizes each batch to stabilize and speed up training
            nn.ReLU(), # Disregard all negtive values
            nn.MaxPool2d(kernel_size=2, stride=2), # Halves spatial dimensions (height and width) by taking the max in each 2×2 region — reduces computation and adds spatial invariance
            nn.Dropout2d(p=0.2), # Spatial dropout
        )

        
        # Transformer part
        """The transformer uses 8 attention heads and 2 encoder layers andn dropout for regularization"""
        encoder_layer = nn.TransformerEncoderLayer(d_model=32, nhead=8, dropout=dropout, batch_first=True)
        """Embedding Dimension must matched the ouput feature vector from the CNN layer, in this case, 32"""
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.dropout = nn.Dropout(p=dropout)

        # Linear classifier
        """Final fully connected layer mapping the 32-dimensional feature vector to num_classes (32) output logits — one score per possible key."""
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        """"Input the spectrogram samples in a batch (batch, 1, height, width)"""
        x = self.conv_layers(x)
        """Resulting output of (batch, 32, height/2, width/2"""
        
        # Flattening
        """Reshapes for Transformer input. Transformer expects (sequence_length, batch, features) or (batch, sequence_length, features)"""
        x = x.view(x.size(0), -1, x.size(1))

        # Transformer encoder
        """Passing through the 2 encoder layers"""
        x = self.transformer_encoder(x)

        """Global max pooling over the sequence dimension — collapses (batch, seq_len, 32) → (batch, 32)"""
        # Max pooling over time
        x, _ = torch.max(x, dim=1)

        """Apply dropout before final classification"""
        x = self.dropout(x) 

        """Maps (batch, 32) → (batch, 36) — raw logit scores for each class"""
        # Classifier
        x = self.fc(x)
        return x

    def get_num_params(self):
        """Return the number of trainable parameters"""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

### Training and Validation

In [ ]:
# Set device (MPS for Mac)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS (Metal Performance Shaders)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

# Move model to device
AudioModel = CoAtNet().to(device)
print(f"Models moved to {device}")

# Training hyperparameters
LEARNING_RATE = 5e-4
NUM_EPOCHS = 1100

print("\n" + "=" * 60)
print("TRAINING CONFIGURATION FOR Combined Audio data loader")
print("=" * 60)
print(f"Learning rate: {LEARNING_RATE}")
print(f"Number of epochs: {NUM_EPOCHS}")
print(f"Batch size: {AudioTrainLoader.batch_size}")
print(f"Device: {device}")
print(f"Total training batches per epoch: {len(AudioTrainLoader)}")
print(f"Total validation batches per epoch: {len(AudioValLoader)}")




# Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()

AudioOptimizer = optim.Adam(AudioModel.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# LR scheduler

"""ReduceLROnPlateau — cuts LR when val F1 stops improving"""
AudioScheduler = optim.lr_scheduler.ReduceLROnPlateau(
    AudioOptimizer,
    mode     = 'max',      # Maximize F1
    factor   = 0.5,        # Halve the LR
    patience = 50         # Wait 50 epochs before reducing
)

In [ ]:
def plot_confusion_matrix(cm, class_names, title, num_classes):
    fig, ax = plt.subplots(figsize=(16, 14))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=14, pad=20)

    tick_marks = np.arange(num_classes)
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(class_names, fontsize=8, rotation=45, ha='right')
    ax.set_yticklabels(class_names, fontsize=8)

    thresh = cm.max() / 2.0
    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(
                j, i, format(cm[i, j], 'd'),
                ha='center', va='center', fontsize=7,
                color='white' if cm[i, j] > thresh else 'black',
                fontweight='bold' if i == j else 'normal'   # Bold on diagonal = correct predictions
            )

    ax.set_ylabel('True Label',      fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()


def plot_f1_per_class(f1_per_class, class_names, title, num_classes):
    """Bar chart showing F1 score per keystroke class."""
    fig, ax  = plt.subplots(figsize=(16, 5))
    colors   = plt.cm.RdYlGn(f1_per_class)           # Red=low, Green=high
    bars     = ax.bar(range(num_classes), f1_per_class, color=colors)

    ax.set_xlabel('Keystroke Class', fontsize=12)
    ax.set_ylabel('F1 Score',        fontsize=12)
    ax.set_title(title,              fontsize=14)
    ax.set_xticks(range(num_classes))
    ax.set_xticklabels(class_names,  fontsize=9, rotation=45, ha='right')
    ax.set_ylim([0, 1])
    ax.axhline(y=np.mean(f1_per_class), color='blue',
               linestyle='--', linewidth=1.5,
               label=f'Mean F1: {np.mean(f1_per_class):.3f}')
    ax.grid(axis='y', alpha=0.3)
    ax.legend()

    # Value label on each bar
    for bar, f1 in zip(bars, f1_per_class):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height(),
                f'{f1:.2f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

    plt.tight_layout()
    plt.show()


def print_metrics_table(precision_per_class, recall_per_class,
                        f1_per_class, support_per_class,
                        precision_weighted, recall_weighted,
                        f1_weighted, accuracy, class_names, num_classes, title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")
    print(f"  Accuracy  : {accuracy:.4f}")
    print(f"  Precision : {precision_weighted:.4f}  (weighted)")
    print(f"  Recall    : {recall_weighted:.4f}  (weighted)")
    print(f"  F1 Score  : {f1_weighted:.4f}  (weighted)")

    print(f"\n{'Per-Class Metrics':}")
    print(f"{'='*70}")
    print(f"  {'Class':<10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print(f"  {'-'*50}")
    for i in range(num_classes):
        print(f"  {class_names[i]:<10} "
              f"{precision_per_class[i]:>10.4f} "
              f"{recall_per_class[i]:>10.4f} "
              f"{f1_per_class[i]:>10.4f} "
              f"{support_per_class[i]:>10}")
    print(f"  {'-'*50}")
    print(f"  {'Weighted':<10} "
          f"{precision_weighted:>10.4f} "
          f"{recall_weighted:>10.4f} "
          f"{f1_weighted:>10.4f} "
          f"{sum(support_per_class):>10}")
    print(f"{'='*70}")



In [ ]:
def train(model, train_loader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Progress bar
    progress_bar = tqdm(train_loader, desc='Training', leave=False)
    
    for waveforms, labels in progress_bar:
        # Move data to device
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()  # Clear previous gradients
        logits = model(waveforms)
        loss = criterion(logits, labels)
        
        # Backward pass
        loss.backward()  # Calculate gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Prevent exploding gradients
        optimizer.step()  # Update weights
        
        # Track metrics
        total_loss += loss.item() # Useful for visualizing the convergence time
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Update progress bar
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        if loss.item() < 0.0001:
            break
    
    avg_loss = total_loss / len(train_loader)
    accuracy = accuracy_score(all_labels, all_preds)   
    return avg_loss, accuracy


def validate(model, val_loader, criterion, device, num_classes, show_plots=False, epoch=None, model_name=""):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # No gradient calculation needed for validation
    with torch.no_grad():
        for waveforms, labels in tqdm(val_loader, desc='Validation', leave=False):
            # Move data to device
            waveforms = waveforms.to(device)
            labels = labels.to(device)
            
            # Forward pass only
            logits = model(waveforms)
            loss = criterion(logits, labels)
            
            # Track metrics
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(val_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(all_labels, all_preds, average=None, zero_division=0, labels=list(range(num_classes)))
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
    epoch_label = f"Epoch {epoch}" if epoch is not None else "Validation"
    print_metrics_table(
        precision_per_class, recall_per_class, f1_per_class, support_per_class,
        precision_weighted,  recall_weighted,  f1_weighted,  accuracy,
        CLASS_NAMES, num_classes,
        title=f"Validation Results — {model_name} | {epoch_label}"
    )

    if show_plots:
        plot_confusion_matrix(
            cm, CLASS_NAMES,
            title=f"Confusion Matrix — {model_name} | {epoch_label}",
            num_classes=num_classes
        )
        plot_f1_per_class(
            f1_per_class, CLASS_NAMES,
            title=f"Per-Class F1 — {model_name} | {epoch_label}",
            num_classes=num_classes
        )

    return avg_loss, accuracy, precision_per_class, recall_per_class, f1_per_class, support_per_class, precision_weighted, recall_weighted, f1_weighted


def test(model, test_loader, device, model_name, num_classes):
    model.eval()
    all_preds  = []
    all_labels = []

    print("\n" + "=" * 60)
    print(f"  EVALUATING {model_name} ON TEST SET")
    print("=" * 60)

    with torch.no_grad():
        for waveforms, labels in tqdm(test_loader, desc='Testing'):
            waveforms = waveforms.to(device)
            labels    = labels.to(device)

            logits = model(waveforms)
            preds  = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())


    accuracy = accuracy_score(all_labels, all_preds)
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0, labels=list(range(num_classes)))
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))


    print_metrics_table(
        precision_per_class, recall_per_class, f1_per_class, support_per_class,
        precision_weighted,  recall_weighted,  f1_weighted,  accuracy,
        CLASS_NAMES, num_classes,
        title=f"Test Results — {model_name}"
    )

    print("\nDetailed Classification Report:")
    print("-" * 60)
    print(classification_report(
        all_labels, all_preds,
        target_names=CLASS_NAMES,
        digits=4
    ))


    plot_confusion_matrix(
        cm, CLASS_NAMES,
        title=f"Confusion Matrix — {model_name} | Test Set",
        num_classes=num_classes
    )
    plot_f1_per_class(
        f1_per_class, CLASS_NAMES,
        title=f"Per-Class F1 — {model_name} | Test Set",
        num_classes=num_classes
    )


def plot_training_history(history, model_name, model, best_val_f1):
    fig, axes = plt.subplots(2, 3, figsize=(10, 7))
    epochs    = range(1, len(history['train_loss']) + 1)

    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[0, 0].plot(epochs, history['val_loss'],   'r-', label='Val Loss',   linewidth=2)
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
    axes[0, 1].plot(epochs, history['val_acc'],   'r-', label='Val Acc',   linewidth=2)
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

    axes[0, 2].plot(epochs, history['weighted_val_f1'], 'g-', linewidth=2)
    axes[0, 2].set_xlabel('Epoch'); axes[0, 2].set_ylabel('Weighted F1')
    axes[0, 2].set_title('Weighted Validation F1 Score')
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].axhline(y=best_val_f1, color='r', linestyle='--', label=f'Best: {best_val_f1:.4f}')
    axes[0, 2].legend()

    axes[1, 0].plot(epochs, history['weighted_val_precision'], 'b-', label='Precision', linewidth=2)
    axes[1, 0].plot(epochs, history['weighted_val_recall'],    'r-', label='Recall',    linewidth=2)
    axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Score')
    axes[1, 0].set_title('Weighted Validation Precision & Recall')
    axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(epochs, history['learning_rates'], 'purple', linewidth=2)  # Fix: now populated
    axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].grid(True, alpha=0.3); axes[1, 1].set_yscale('log')

    axes[1, 2].axis('off')
    summary_text = f"""
    Training Summary — {model_name}
    ================================
    Best Weighted Val F1 : {best_val_f1:.4f}
    Final Train Acc      : {history['train_acc'][-1]:.4f}
    Final Val Acc        : {history['val_acc'][-1]:.4f}
    Final Val Precision  : {history['weighted_val_precision'][-1]:.4f}
    Final Val Recall     : {history['weighted_val_recall'][-1]:.4f}
    Total Epochs         : {len(history['train_loss'])}
    Model Parameters     : {model.get_num_params():,}
    """
    axes[1, 2].text(0.05, 0.5, summary_text, fontsize=11,
                    family='monospace', verticalalignment='center')

    plt.tight_layout()
    plt.show()


class EarlyStopping:
    """Detect when F1 score is no longer improving or plaeteau after certain epochs during training"""
    def __init__(self, patience=100, min_delta=0.001):
        self.patience   = patience # How many epochs of not improving to be tolerated, eg F1 score not improving after 100 epochs ->  Signal stop training
        self.min_delta  = min_delta # Initial minimum data
        self.best_score = 0 # Best F1 score to keep track and compare
        self.counter    = 0 # Counting how many epochs have the F1 score not improving
        self.should_stop = False # Signal Stop training or not

    """Take in current F1 score and compare with the previous best F1 score to check whether it is improving or not. If not for 100 epochs, terminate signal is send to stop  wasting time and resource"""
    def step(self, val_f1):
        if val_f1 > self.best_score + self.min_delta:
            self.best_score = val_f1
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"Early stopping — no improvement for {self.patience} epochs")


### Training And Validation Loop

In [ ]:
history = {
    "Combined Audio": {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'weighted_val_precision': [],
        'weighted_val_recall': [],
        'weighted_val_f1': [],
        'learning_rates': []           
    }
}

PLOT_EVERY_N_EPOCHS = 1


best_val_f1_tracker = {}


best_val_f1 = 0
best_model_state = None
model = AudioModel
train_loader = AudioTrainLoader
val_loader = AudioValLoader
optimizer = AudioOptimizer
scheduler = AudioScheduler
# earlyStopper = EarlyStopping(patience=200)

print("\n" + "=" * 60)
print(f"  STARTING TRAINING: Combined Dataset")
print("=" * 60)
print(f"  Epochs     : {NUM_EPOCHS}")
print(f"  Device     : {device}")
print(f"  Parameters : {model.get_num_params():,}")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 60)

    """Training Phrase"""
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)

    """Validating Phrase"""
    show_plots = ((epoch + 1) % PLOT_EVERY_N_EPOCHS == 0) or (epoch + 1 == NUM_EPOCHS)
    (val_loss, val_acc, precision_per_class, recall_per_class, f1_per_class, support_per_class, precision_weighted, recall_weighted, f1_weighted) = validate(model, val_loader, criterion, device, num_classes = 36, show_plots = show_plots, epoch = epoch + 1, model_name = "Combined Audio")

    scheduler.step(f1_weighted) # Update LR based on val F1
    # earlyStopper.step(f1_weighted) # Update F1 score to check for plateau
        
    """Logging History"""
    current_lr = optimizer.param_groups[0]['lr']
    history["Combined Audio"]['train_loss'].append(train_loss)
    history["Combined Audio"]['train_acc'].append(train_acc)
    history["Combined Audio"]['val_loss'].append(val_loss)
    history["Combined Audio"]['val_acc'].append(val_acc)
    history["Combined Audio"]['weighted_val_precision'].append(precision_weighted)
    history["Combined Audio"]['weighted_val_recall'].append(recall_weighted)
    history["Combined Audio"]['weighted_val_f1'].append(f1_weighted)
    history["Combined Audio"]['learning_rates'].append(current_lr)

    """Saving Best Model"""
    if f1_weighted > best_val_f1:
        best_val_f1 = f1_weighted
        best_model_state = model.state_dict().copy()
        print(f"  ★ New best Weighted F1: {f1_weighted:.4f}")

    epoch_time = time.time() - start_time
    print(f"  Time      : {epoch_time:.2f}s")
    print(f"  Train     → Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"  Val       → Loss: {val_loss:.4f}   | Acc: {val_acc:.4f}")
    print(f"  Weighted  → P: {precision_weighted:.4f} | R: {recall_weighted:.4f} | F1: {f1_weighted:.4f}")
        
    # if earlyStopper.should_stop:
        # break
    
"""Load Best Weights"""
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✓ Loaded best weights  (F1: {best_val_f1:.4f})")

best_val_f1_tracker["Combined Audio"] = best_val_f1

In [ ]:
plot_training_history(history["Combined Audio"], "Phone Audio CoAtNet", model, best_val_f1_tracker["Combined Audio"])

### Save Model

In [ ]:
model_save_path = f'CombinedAudio.pth'
torch.save(model.state_dict(), model_save_path)
print(f"\n✓ Model saved to: {model_save_path}")